In [ ]:
import json
import shutil
from pathlib import Path
from time import sleep

import lancedb
import pandas as pd
from loguru import logger

from dreamai.ai import ModelName
from dreamai.rag import application
from dreamai.rag_utils import add_data_with_descriptions
from dreamai.settings import CreatorSettings, RAGAppSettings, RAGSettings
from dreamai.utils import flatten_list

%load_ext autoreload
%autoreload 2
%reload_ext autoreload

In [ ]:
creator_settings = CreatorSettings()
rag_settings = RAGSettings()
rag_app_settings = RAGAppSettings()

RERANKER = rag_settings.reranker
HAS_WEB = rag_app_settings.has_web
MODEL = ModelName.GEMINI_FLASH
LANCE_URI = "lance/RFP"
DATA = "/media/hamza/data2/RFP/docs/"

if Path(LANCE_URI).exists():
    shutil.rmtree(LANCE_URI)

lance_db = lancedb.connect(uri=LANCE_URI)

table_descriptions = add_data_with_descriptions(model=MODEL, lance_db=lance_db, data=DATA)

In [ ]:
df = pd.read_csv("RFP.csv")
questions = df.iloc[:, 0].tolist()

In [ ]:
app = application(db=lance_db, reranker=None, model=MODEL, has_web=False, only_data=True)

In [ ]:
qna = {"questions": questions[-8:-6], "answers": [], "sources": []}

In [ ]:
qna

In [ ]:
for query in qna["questions"]:
    inputs = {"query": query}
    logger.info(f"\nProcessing query: {query}")
    while True:
        step_result = app.step(inputs=inputs)
        if step_result is None:
            logger.error("Error: app.step() returned None")
            break
        action, result, _ = step_result
        logger.info(f"\nAction: {action.name}\n")
        logger.success(f"RESULT: {result}\n")
        if action.name == "terminate":
            break
        elif action.name in ["update_chat_history"]:
            qna["answers"].append(result["chat_history"][-1]["content"].split("\n"))
            qna["sources"].append(
                [
                    {k: v for k, v in json.loads(d).items() if k != "index"}
                    for d in set(
                        [
                            json.dumps(s, sort_keys=True)
                            for s in flatten_list(app.state.get("source_docs", []))
                        ]
                    )
                ]
            )
            sleep(1)
            break